# Feature Engineering
Building meaningful new features from the cleaned dataset.
These features capture business logic that raw columns can't express on their own.

In [1]:
import pandas as pd
import numpy as np

# load cleaned dataset
df = pd.read_csv('../data/processed/telco_cleaned.csv')

print("Shape:", df.shape)
df.head(3)

Shape: (7043, 21)


,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,...,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value,CLTV
0,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,3239
1,Female,No,No,Yes,2,Yes,No,Fiber optic,No,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,2701
2,Female,No,No,Yes,8,Yes,Yes,Fiber optic,No,No,...,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1,5372


### 1. Tenure Groups
Grouping customers by how long they've been with the company.
New customers behave very differently from loyal long-term customers.

In [2]:
# grouping tenure into meaningful business segments
# 0-12 months: new, 12-24: developing, 24-48: established, 48+: loyal
def tenure_group(tenure):
    if tenure <= 12:
        return 'New'
    elif tenure <= 24:
        return 'Developing'
    elif tenure <= 48:
        return 'Established'
    else:
        return 'Loyal'

df['Tenure Group'] = df['Tenure Months'].apply(tenure_group)

print("Tenure Group Distribution:")
print(df['Tenure Group'].value_counts())
print("\nChurn Rate by Tenure Group:")
print(df.groupby('Tenure Group')['Churn Value'].mean().mul(100).round(1).astype(str) + '%')

Tenure Group Distribution:
Tenure Group
Loyal          2239
New            2186
Established    1594
Developing     1024
Name: count, dtype: int64

Churn Rate by Tenure Group:
Tenure Group
Developing     28.7%
Established    20.4%
Loyal           9.5%
New            47.4%
Name: Churn Value, dtype: str


### 2. Spending Features
Creating features that capture how much a customer spends 
and how their spending compares to their tenure.

In [3]:
# average monthly spend over entire tenure
# captures if customer is spending more or less than their monthly charges suggest
df['Avg Monthly Spend'] = df['Total Charges'] / (df['Tenure Months'] + 1)

# how much customer pays relative to their CLTV
# high ratio means they are paying a lot compared to their predicted value
df['Charge to CLTV Ratio'] = df['Monthly Charges'] / df['CLTV']

# total services subscribed - counts how many add-ons a customer has
service_cols = ['Phone Service', 'Multiple Lines', 'Online Security', 
                'Online Backup', 'Device Protection', 'Tech Support', 
                'Streaming TV', 'Streaming Movies']

df['Total Services'] = df[service_cols].apply(lambda x: (x == 'Yes').sum(), axis=1)

print("New Features Sample:")
print(df[['Avg Monthly Spend', 'Charge to CLTV Ratio', 'Total Services']].describe().round(2))

New Features Sample:
       Avg Monthly Spend  Charge to CLTV Ratio  Total Services
count            7043.00               7043.00         7043.00
mean               58.99                  0.02            3.36
std                30.58                  0.01            2.06
min                 0.00                  0.00            0.00
25%                26.04                  0.01            1.00
50%                60.94                  0.02            3.00
75%                84.83                  0.02            5.00
max               118.97                  0.05            8.00


 ### 3. Risk Indicators
Creating binary flags that directly signal high churn risk
based on patterns we discovered during EDA.

In [4]:
# binary risk flags based on EDA findings
# month-to-month contract was strongest churn driver at 42.7%
df['Is Month to Month'] = (df['Contract'] == 'Month-to-month').astype(int)

# electronic check had highest churn rate at 45.3%
df['Is Electronic Check'] = (df['Payment Method'] == 'Electronic check').astype(int)

# fiber optic had 41.9% churn rate
df['Is Fiber Optic'] = (df['Internet Service'] == 'Fiber optic').astype(int)

# senior citizens churn at 41.7%
df['Is Senior'] = (df['Senior Citizen'] == 'Yes').astype(int)

# no tech support = 41.6% churn
df['No Tech Support'] = (df['Tech Support'] == 'No').astype(int)

# no online security = 41.8% churn
df['No Online Security'] = (df['Online Security'] == 'No').astype(int)

print("Risk Indicators Distribution:")
risk_cols = ['Is Month to Month', 'Is Electronic Check', 'Is Fiber Optic', 
             'Is Senior', 'No Tech Support', 'No Online Security']

for col in risk_cols:
    pct = df[col].mean() * 100
    print(f"{col}: {pct:.1f}% of customers")

Risk Indicators Distribution:
Is Month to Month: 55.0% of customers
Is Electronic Check: 33.6% of customers
Is Fiber Optic: 44.0% of customers
Is Senior: 16.2% of customers
No Tech Support: 71.0% of customers
No Online Security: 71.3% of customers


### 4. Customer Value Segment
Segmenting customers by CLTV into value tiers.
Helps prioritize retention efforts on high value customers.

In [5]:
# segmenting customers into value tiers based on CLTV
# using quartiles to define segments
cltv_25 = df['CLTV'].quantile(0.25)
cltv_75 = df['CLTV'].quantile(0.75)

def value_segment(cltv):
    if cltv <= cltv_25:
        return 'Low Value'
    elif cltv <= cltv_75:
        return 'Medium Value'
    else:
        return 'High Value'

df['Value Segment'] = df['CLTV'].apply(value_segment)

print(f"CLTV Quartiles: 25%={cltv_25:.0f}, 75%={cltv_75:.0f}")
print("\nValue Segment Distribution:")
print(df['Value Segment'].value_counts())
print("\nChurn Rate by Value Segment:")
print(df.groupby('Value Segment')['Churn Value'].mean().mul(100).round(1).astype(str) + '%')

CLTV Quartiles: 25%=3469, 75%=5380

Value Segment Distribution:
Value Segment
Medium Value    3519
Low Value       1763
High Value      1761
Name: count, dtype: int64

Churn Rate by Value Segment:
Value Segment
High Value      20.7%
Low Value       34.4%
Medium Value    25.5%
Name: Churn Value, dtype: str


### 5. Encode Categorical Columns
Converting text columns to numbers so machine learning models can process them.

In [6]:
# binary encoding for yes/no columns
binary_cols = ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service',
               'Multiple Lines', 'Online Security', 'Online Backup', 'Device Protection',
               'Tech Support', 'Streaming TV', 'Streaming Movies', 'Paperless Billing']

for col in binary_cols:
    df[col] = (df[col].str.strip() == 'Yes').astype(int)

# one hot encoding for multi-value columns
df = pd.get_dummies(df, columns=['Internet Service', 'Contract', 
                                  'Payment Method', 'Tenure Group', 
                                  'Value Segment'], drop_first=False)

print("Shape after encoding:", df.shape)
print("\nAll Columns:")
print(df.columns.tolist())

Shape after encoding: (7043, 44)

All Columns:
['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Paperless Billing', 'Monthly Charges', 'Total Charges', 'Churn Value', 'CLTV', 'Avg Monthly Spend', 'Charge to CLTV Ratio', 'Total Services', 'Is Month to Month', 'Is Electronic Check', 'Is Fiber Optic', 'Is Senior', 'No Tech Support', 'No Online Security', 'Internet Service_DSL', 'Internet Service_Fiber optic', 'Internet Service_No', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'Payment Method_Bank transfer (automatic)', 'Payment Method_Credit card (automatic)', 'Payment Method_Electronic check', 'Payment Method_Mailed check', 'Tenure Group_Developing', 'Tenure Group_Established', 'Tenure Group_Loyal', 'Tenure Group_New', 'Value Segment_High Value', 'Value Segment_Low Value', 'Value Segment_Medium Value

In [10]:
# converting all boolean columns to integers (True/False -> 1/0)
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

# verify no boolean columns remain
print("Remaining boolean columns:", df.select_dtypes(include='bool').columns.tolist())
print("\nSample of fixed columns:")
print(df[bool_cols].head(3))

Remaining boolean columns: []

Sample of fixed columns:
Empty DataFrame
Columns: []
Index: [0, 1, 2]


In [12]:
# saving final engineered dataset
df.to_csv('../data/processed/telco_engineered.csv', index=False)

print("Final dataset saved successfully.")
print("Shape:", df.shape)
print("\nTarget variable distribution:")
print(df['Churn Value'].value_counts())

Final dataset saved successfully.
Shape: (7043, 44)

Target variable distribution:
Churn Value
0    5174
1    1869
Name: count, dtype: int64


## Summary
- Built 6 new features: Tenure Group, Avg Monthly Spend, Charge to CLTV Ratio, Total Services, Value Segment, Risk Indicators
- Encoded all categorical columns using binary and one-hot encoding
- Final dataset: 7043 rows, 44 features
- Dataset saved to data/processed/telco_engineered.csv